# 01 · Data Collection

The full dataset (`data/raw/products.jsonl`, 4,207 products from 8 sources) was already scraped in the
`product_search_and_analysis` project. This notebook is a **sample** of how that collection works, for one
source: **ptc-computer.com.kh**, a computer shop in Cambodia.

Steps: download a listing page → parse the product cards → follow pagination → save in the products format.

Scraping rules used here:
- identify the bot with a `User-Agent`
- wait `DELAY_SECONDS` between requests
- only scrape a few pages (`MAX_PAGES`)
- save the sample to a separate file, never over `products.jsonl`

In [ ]:
import json
import re
import time
from pathlib import Path
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
SAMPLE_OUTPUT = ROOT / "data" / "raw" / "sample_ptc_computer.jsonl"

SOURCE = "ptc_computer"
LISTINGS = {  # our category -> listing page on the site
    "monitor": "https://ptc-computer.com.kh/parent_category/monitor",
    "printer": "https://ptc-computer.com.kh/parent_category/printer-and-cartdridges",
}
MAX_PAGES = 2          # pages per listing (sample only)
DELAY_SECONDS = 2.0    # pause before every request
USER_AGENT = "CatalogSearch-ResearchBot/0.1 (academic information-retrieval project)"
HEADERS = {"User-Agent": USER_AGENT, "Accept-Language": "en-US,en;q=0.9"}

pd.set_option("display.max_colwidth", 60)

## 1. Download a page

In [ ]:
def download(url):
    """GET a page after a polite delay. Returns the HTML, or None on any error."""
    time.sleep(DELAY_SECONDS)
    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
    except requests.RequestException as error:
        print(f"network error on {url}: {error}")
        return None
    if response.status_code != 200:
        print(f"HTTP {response.status_code} on {url}")
        return None
    return response.content.decode("utf-8", errors="replace")


first_url = LISTINGS["monitor"]
html = download(first_url)
print(f"downloaded {len(html or '')} characters from {first_url}")

## 2. Parse the product cards
Each product on a listing page is a `.product-card`. The card already holds the title, price, brand, image and a
short spec list, so product pages don't need to be opened.

In [ ]:
PRICE_RE = re.compile(r"\$\s?([\d,]+(?:\.\d+)?)")                # "$1,889" -> 1889
PRODUCT_NUMBER_RE = re.compile(r"\((\d+)")                       # "addToCompare(3856)" -> 3856
SPEC_RE = re.compile(r"^([A-Za-z][\w /&().+-]{0,30}):\s*(.+)$")  # "CPU: Intel Core i5" -> key, value


def text_of(element):
    return element.get_text(" ", strip=True) if element else ""


def parse_price(text):
    match = PRICE_RE.search(text or "")
    return float(match.group(1).replace(",", "")) if match else None


def spec_lines(card):
    """The card's spec list as clean lines, e.g. ['Size: 27 inch', 'Resolution: 2560x1440']."""
    block = card.select_one("#product_sort_description") or card.select_one("h3:nth-of-type(2) a")
    if not block:
        return []
    for br in block.find_all("br"):
        br.replace_with("\n")
    lines = [line.strip(" -–\t") for line in block.get_text("").splitlines()]
    return [re.sub(r"\s+", " ", line) for line in lines if line.strip(" -–\t")]


def parse_brand(card, title):
    """Brand logo link '/brand/dell-laptop-desktop-cambodia' -> 'Dell'."""
    link = card.select_one('a[href*="/brand/"]')
    if not link:
        return None
    slug_word = link["href"].rstrip("/").rsplit("/", 1)[-1].split("-")[0]
    return next((w for w in title.split() if w.lower() == slug_word.lower()), slug_word.capitalize())


def parse_card(card, category):
    """One product card -> one product in the products format (12 fields)."""
    title_link = card.select_one("h3 a")
    if not title_link:
        return None
    title = text_of(title_link)
    url = title_link["href"]

    compare = card.select_one('[onclick*="addToCompare"]')
    number = PRODUCT_NUMBER_RE.search(compare["onclick"]) if compare else None
    product_key = number.group(1) if number else url.rstrip("/").rsplit("/", 1)[-1]

    price_element = card.select_one('span.fw-600[style*="18px"]')
    image = card.select_one(".product_gallery img") or card.select_one("img.img-fit")
    lines = spec_lines(card)
    specs = {m.group(1).strip(): m.group(2).strip() for m in map(SPEC_RE.match, lines) if m}
    specs["Stock"] = text_of(card.select_one(".text-capitalize"))

    return {
        "product_id": f"{SOURCE}:{product_key}",
        "title": title,
        "description": " | ".join(lines) or title,
        "brand": parse_brand(card, title),
        "category": category,
        "price": parse_price(text_of(price_element)),
        "specs": {k: v for k, v in specs.items() if v},
        "features": lines,
        "image_url": (image.get("data-src") or image.get("src")) if image else None,
        "source_url": url,
        "rating": None,  # the site shows no real ratings
        "source": SOURCE,
    }


def parse_listing_page(html, category, page_url):
    """Listing page -> (products, URL of the next page or None on the last page)."""
    soup = BeautifulSoup(html, "html.parser")
    products = [p for p in (parse_card(card, category) for card in soup.select(".product-card")) if p]
    next_link = soup.select_one('.pagination a[rel="next"]')
    return products, (urljoin(page_url, next_link["href"]) if next_link else None)


page_products, next_url = parse_listing_page(html or "", "monitor", first_url)
print(f"{len(page_products)} products on the first page; next page: {next_url}")
page_products[0] if page_products else "no products parsed: the site layout may have changed"

## 3. Scrape the sample (follow pagination)

In [ ]:
def scrape(listings, max_pages=MAX_PAGES):
    products, seen = [], set()
    for category, url in listings.items():
        for page in range(1, max_pages + 1):
            if not url:
                break
            html = download(url)
            if html is None:
                break
            page_products, url = parse_listing_page(html, category, url)
            new = [p for p in page_products if p["product_id"] not in seen]
            seen.update(p["product_id"] for p in new)
            products.extend(new)
            print(f"{category}: page {page}, {len(new)} products")
            if not page_products:
                break
    return products


sample = scrape(LISTINGS)
sample_df = pd.DataFrame(sample)
print(f"\n{len(sample)} products scraped")
sample_df[["product_id", "title", "brand", "category", "price"]].head(10)

## 4. Save the sample

In [ ]:
SAMPLE_OUTPUT.parent.mkdir(parents=True, exist_ok=True)
with open(SAMPLE_OUTPUT, "w", encoding="utf-8") as f:
    for product in sample:
        f.write(json.dumps(product, ensure_ascii=False) + "\n")
print(f"saved {len(sample)} products to {SAMPLE_OUTPUT.relative_to(ROOT)}")

## 5. The full collected dataset
The sample above uses the same 12 fields as the full dataset collected in `product_search_and_analysis`.

In [ ]:
full = pd.read_json(ROOT / "data" / "raw" / "products.jsonl", lines=True)
print(f"{len(full)} products, {full['category'].nunique()} categories")
print("same columns as the sample:", list(full.columns) == list(sample_df.columns) if len(sample_df) else "n/a")
full["source"].value_counts().rename("products").to_frame()